# Notebook 7: Backbone ablations: fusion of the three modality encoders and visit pooling

The Notebook 2 backbone encodes diagnoses, medications and procedures with three parallel Transformers and
concatenates their outputs: modalities meet only in the final linear layer, so the logit of a label is a
sum of a diagnosis term, a medication term and a procedure term. Conjunctions ("epilepsy *and* stiripentol")
can only be approximated additively. Within a visit, codes are summed, so a rare code's embedding is mixed
with the 11.7 diagnoses, 1.6 procedures and 18.1 medication classes of an average admission (Notebook 1).

| config | fusion | pooling | factor tested |
|---|---|---|---|
| `concat_sum` | concatenation $[h_{dx}; h_{rx}; h_{px}]$ | sum | Notebook 2 |
| `sum_sum` | $h_{dx}+h_{rx}+h_{px}$ | sum | does keeping modalities separate matter? |
| `cross_sum` | one attention layer over the visit states of all modalities | sum | cross-modality interactions |
| `concat_set` | concatenation | attention pooling (Set Transformer block) | pooling dilution (Hypothesis B) |
| `cross_set` | cross | attention pooling | both |

Same protocol as Notebook 6.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path(os.environ.get('RAREPRED_ROOT', '..')).resolve()
sys.path.insert(0, str(ROOT / 'src'))
PROCESSED   = Path(os.environ.get('RAREPRED_PROCESSED', ROOT / 'processed'))
EXTERNAL    = ROOT / 'data' / 'external'
RESULTS     = ROOT / 'results'
FIGURES     = ROOT / 'figures'
CHECKPOINTS = ROOT / 'checkpoints'
for d in (RESULTS, FIGURES, CHECKPOINTS):
    d.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, torch
from rarepred.experiment import Setup, run_config
from rarepred.metrics import paired_bootstrap_delta
SEED       = int(os.environ.get('RAREPRED_SEED', 42))
MAX_EPOCHS = int(os.environ.get('RAREPRED_MAX_EPOCHS', 50))
RETRAIN    = os.environ.get('RAREPRED_RETRAIN', '0') == '1'
RUN        = [r for r in os.environ.get('RAREPRED_RUN', '').split(',') if r]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', DEVICE, '| seed', SEED, '| max epochs', MAX_EPOCHS)

RUN_DIR = 'backbone_ablations'

import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 9, 'axes.titlesize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#666666', 'axes.labelcolor': '#191919', 'xtick.color': '#666666',
    'ytick.color': '#666666', 'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'legend.frameon': False,
})
BLUE, ORANGE, AQUA, GREY = '#257de0', '#f47230', '#00b894', '#9c9c9c'

In [ ]:
S = Setup(PROCESSED, DEVICE,
          max_train=int(os.environ['RAREPRED_MAX_TRAIN']) if os.environ.get('RAREPRED_MAX_TRAIN') else None)
print(f"max |logit diff| default backbone vs Notebook 2 SharedBackbone: {S.equivalence_check():.2e}")
CONFIGS = {
    'concat_sum': dict(fusion='concat', pooling='sum'),
    'sum_sum':    dict(fusion='sum', pooling='sum'),
    'cross_sum':  dict(fusion='cross', pooling='sum'),
    'concat_set': dict(fusion='concat', pooling='set'),
    'cross_set':  dict(fusion='cross', pooling='set'),
}
names = RUN or list(CONFIGS)

def run_all(configs, run_names):
    outs, preds = {}, {}
    for name in run_names:
        out, probs, ys, _ = run_config(S, name, configs[name], CHECKPOINTS, RESULTS / RUN_DIR,
                                       max_epochs=MAX_EPOCHS, seed=SEED, retrain=RETRAIN)
        outs[name], preds[name] = out, probs
    return outs, preds, ys

def summary_table(outs):
    rows = []
    for name, o in outs.items():
        r, p, mg = o['test_sample_quartiles'], o['test_patient_strata'], o['test_medco_mass_groups']
        rows.append({'config': name,
                     'sample-AUPRC': r['sample_auprc'],
                     'Q1 (rarest)': r['sample_auprc[Q1 (rarest)]'], 'Q2': r['sample_auprc[Q2]'],
                     'Q3': r['sample_auprc[Q3]'], 'Q4 (common)': r['sample_auprc[Q4 (common)]'],
                     'gap Q4-Q1': r['gap_sample_auprc'],
                     'label-AUPRC rare': p['label_auprc[rare]'], 'label-AUPRC common': p['label_auprc[common]'],
                     'MedCo-groups 0-25%': mg['sample_auprc[Q1 (rarest)]'],
                     'MedCo-groups 75-100%': mg['sample_auprc[Q4 (common)]'],
                     'acc@m': r[f"acc@{S.meta['m_median']}"]})
    return pd.DataFrame(rows).set_index('config').round(4)

outs, preds, ys = run_all(CONFIGS, names)
table = summary_table(outs)
table['params'] = [outs[n]['n_params'] for n in table.index]
table

In [ ]:
q1 = S.groups_sample['Q1 (rarest)']
if 'concat_sum' in preds:
    rows = []
    for name in preds:
        if name != 'concat_sum':
            d, lo, hi = paired_bootstrap_delta(preds['concat_sum'], preds[name], ys, q1)
            rows.append({'config': name, 'delta Q1 vs concat_sum': d, 'CI low': lo, 'CI high': hi})
    display(pd.DataFrame(rows).set_index('config').round(4))
with open(RESULTS / 'backbone_ablations_summary.json', 'w') as f:
    json.dump({'seed': SEED, 'table': table.reset_index().to_dict(orient='records')}, f, indent=2)